# Análisis Final y Conclusiones

En este notebook se presenta un resumen del flujo completo del proyecto, la revisión de los resultados obtenidos por los modelos base y el modelo optimizado, y la elección del modelo final. Finalmente, se exponen las limitaciones del estudio y recomendaciones futuras.

## 1. Resumen del Flujo Completo
El flujo de trabajo consistió en:
1. **Análisis Exploratorio:** Revisión del dataset, limpieza, PCA y agrupamiento con KMeans (3 clusters).
2. **Modelamiento Supervisado:** Creación de la variable objetivo `is_disrupted` y entrenamiento de modelos base (Regresión Logística, Árbol de Decisión y Random Forest). Se excluyeron variables como `conflict_phase` para evitar *data leakage*.
3. **Validación Cruzada:** Evaluación de la estabilidad de los modelos con la métrica F1-Score.
4. **Optimización de Hiperparámetros:** Ajuste del mejor modelo base (Random Forest) utilizando `GridSearchCV`.

In [4]:
# Carga de librerías necesarias
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Carga de métricas guardadas en los notebooks anteriores
base_cv_metrics = pd.read_csv('../results/metrics/classification_base_vs_cv.csv')
opt_metrics = pd.read_csv('../results/metrics/optimized_model_metrics.csv')

## 2. Revisión de Resultados Principales y Estabilidad
A continuación, revisamos cómo se comportaron los modelos base mediante validación cruzada para responder cuál fue más estable y cuál obtuvo mejor F1 promedio:

In [5]:
# Mostrar tabla de métricas comparativas
display(base_cv_metrics[['model', 'f1_score', 'cv_f1_mean', 'cv_f1_std']])

,model,f1_score,cv_f1_mean,cv_f1_std
0,Logistic Regression,0.892857,0.848412,0.027158
1,Decision Tree,1.000000,0.995556,0.008889
2,Random Forest,1.000000,0.995745,0.008511


Como se observa en la tabla, **Random Forest** y **Decision Tree** obtuvieron los mejores F1 promedio y demostraron ser los modelos más estables (desviación estándar casi nula en validación cruzada). Ambos mantuvieron un rendimiento sumamente alto en comparación con la Regresión Logística.

## 3. Comparación Base vs Optimizado
Posteriormente, se optimizó el Random Forest probando diferentes configuraciones de `n_estimators`, `max_depth` y `min_samples_split`.

In [6]:
# Mostrar resultados del modelo optimizado
display(opt_metrics)

,model,accuracy,precision,recall,f1_score
0,Random Forest Base,1.0,1.0,1.0,1.0
1,Random Forest Optimized,1.0,1.0,1.0,1.0


*(Nota para la presentación: Dado que los árboles base ya arrojaban métricas casi perfectas, la optimización sirve fundamentalmente para controlar la complejidad del árbol —ej. limitando la profundidad— y evitar un posible sobreajuste oculto, aunque numéricamente el F1 se mantenga similar).*

## 4. Elección del Mejor Modelo Final
El modelo final recomendado es el **Random Forest Optimizado**.
- Es un modelo robusto que maneja bien las relaciones complejas del dataset.
- Al definir sus hiperparámetros con GridSearchCV, aseguramos una estructura que generaliza mejor y es menos propensa a memorizar la data.
- Demostró ser el algoritmo más estable a lo largo de todas las particiones de validación cruzada.

## 5. Limitaciones
A pesar de los excelentes resultados predictivos, existen consideraciones importantes:
1. **Resultados extremadamente altos:** Un F1 cercano a 1.0 debe interpretarse con cautela. Es posible que existan variables que aún filtren información del futuro (*data leakage*), o que las reglas de negocio en la data original sean demasiado directas.
2. **Posible desbalance de clases:** La cantidad de vuelos normales vs. los que tienen disrupciones suele estar desbalanceada, lo que influye en el comportamiento del modelo.
3. **Dataset acotado y variables limitadas:** La realidad del tráfico aéreo incluye factores externos (clima, fallas técnicas en tiempo real) que no están en estas columnas.
4. **Simplicidad metodológica:** Para mantener el análisis entendible no se exploraron modelos de ensamble más complejos (ej. XGBoost o LightGBM).

## 6. Recomendaciones Futuras y Conclusiones
**Recomendaciones:**
- Extraer la importancia de características (*Feature Importance*) del modelo final para auditar exactamente qué variables están guiando la predicción perfecta.
- Ampliar el dataset con variables exógenas (datos meteorológicos históricos por ciudad).
- Probar el modelo sobre un conjunto de datos temporalmente distinto (ej. el mes siguiente) para validar la supervivencia del modelo en el tiempo.

**Conclusiones:**
El flujo metodológico confirma que el comportamiento de disrupción de las rutas puede predecirse de manera sumamente estable mediante modelos basados en árboles de decisión (Random Forest). Sin embargo, la perfección de la métrica invita a refinar y complejizar los datos en el futuro para acercarlo aún más a un entorno de producción real en la industria aérea.